# ATC Code Master Table Creation

Retrieve compounds with ATC codes from ChEMBL and save as master tables in Snowflake.

## Tables to Create
1. **ATC_COMPOUND_MAP**: Mapping between compounds and ATC codes (can be joined using InChIKey)
2. **ATC_CLASSIFICATION**: ATC code hierarchy and description master

## Data Source
- ChEMBL (https://www.ebi.ac.uk/chembl/)
- Using chembl_webresource_client

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import logging
from typing import Dict, List, Optional
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ChEMBL client
from chembl_webresource_client.new_client import new_client

# Snowflake
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Project root
project_root = Path.cwd().parent
results_dir = project_root / 'results' / 'atc_master'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Results dir: {results_dir}')

## 1. Retrieve Compounds with ATC Codes from ChEMBL

In [ ]:
# ChEMBL molecule client
molecule = new_client.molecule

# Filter compounds with ATC codes
print('Retrieving compounds with ATC codes from ChEMBL...')
print('(This may take several minutes)')

# Get compounds with ATC codes
atc_molecules = molecule.filter(atc_classifications__isnull=False)

# Select only required fields
atc_molecules = atc_molecules.only([
    'molecule_chembl_id',
    'molecule_structures',
    'atc_classifications',
    'pref_name',
    'max_phase'
])

# Convert to list (handles pagination)
molecules_list = []
for mol in tqdm(atc_molecules, desc='Fetching molecules'):
    molecules_list.append(mol)

print(f'\nRetrieval complete: {len(molecules_list)} compounds')

In [ ]:
# Data formatting: expand to multiple rows if one compound has multiple ATC codes
records = []

for mol in tqdm(molecules_list, desc='Processing molecules'):
    chembl_id = mol.get('molecule_chembl_id')
    structures = mol.get('molecule_structures') or {}
    inchi_key = structures.get('standard_inchi_key')
    smiles = structures.get('canonical_smiles')
    pref_name = mol.get('pref_name')
    max_phase = mol.get('max_phase')
    atc_list = mol.get('atc_classifications') or []
    
    # Skip compounds without InChIKey
    if not inchi_key:
        continue
    
    # Create record for each ATC code
    for atc_code in atc_list:
        if not atc_code or len(atc_code) < 5:
            continue
        
        records.append({
            'CHEMBL_ID': chembl_id,
            'INCHI_KEY': inchi_key,
            'CANONICAL_SMILES': smiles,
            'ATC_CODE': atc_code,
            'ATC_L1': atc_code[0] if len(atc_code) >= 1 else None,
            'ATC_L2': atc_code[:3] if len(atc_code) >= 3 else None,
            'ATC_L3': atc_code[:4] if len(atc_code) >= 4 else None,
            'ATC_L4': atc_code[:5] if len(atc_code) >= 5 else None,
            'PREF_NAME': pref_name,
            'MAX_PHASE': max_phase
        })

df_atc_map = pd.DataFrame(records)

print(f'\nRecords after formatting: {len(df_atc_map):,}')
print(f'Unique compounds: {df_atc_map["CHEMBL_ID"].nunique():,}')
print(f'Unique InChIKeys: {df_atc_map["INCHI_KEY"].nunique():,}')
print(f'Unique ATC codes: {df_atc_map["ATC_CODE"].nunique():,}')

df_atc_map.head(10)

In [ ]:
# Save local backup
df_atc_map.to_parquet(results_dir / 'atc_compound_map.parquet', index=False)
print(f'Saved: {results_dir / "atc_compound_map.parquet"}')

## 2. Retrieve ATC Hierarchy Master

In [ ]:
# ATC classification client
atc_class = new_client.atc_class

print('Retrieving ATC hierarchy master...')

# Get all ATC classes
all_atc = []
for atc in tqdm(atc_class.all(), desc='Fetching ATC classes'):
    all_atc.append(atc)

print(f'\nRetrieval complete: {len(all_atc)} ATC classes')

In [ ]:
# Format ATC hierarchy master
# Each record contains all levels 1-5
# Extract unique entries for each level

atc_records = []
seen_codes = set()

for atc in all_atc:
    # Level 1
    code = atc.get('level1')
    if code and code not in seen_codes:
        seen_codes.add(code)
        atc_records.append({
            'ATC_CODE': code,
            'LEVEL': 1,
            'DESCRIPTION': atc.get('level1_description'),
            'PARENT_CODE': None
        })
    
    # Level 2
    code = atc.get('level2')
    if code and code not in seen_codes:
        seen_codes.add(code)
        atc_records.append({
            'ATC_CODE': code,
            'LEVEL': 2,
            'DESCRIPTION': atc.get('level2_description'),
            'PARENT_CODE': atc.get('level1')
        })
    
    # Level 3
    code = atc.get('level3')
    if code and code not in seen_codes:
        seen_codes.add(code)
        atc_records.append({
            'ATC_CODE': code,
            'LEVEL': 3,
            'DESCRIPTION': atc.get('level3_description'),
            'PARENT_CODE': atc.get('level2')
        })
    
    # Level 4
    code = atc.get('level4')
    if code and code not in seen_codes:
        seen_codes.add(code)
        atc_records.append({
            'ATC_CODE': code,
            'LEVEL': 4,
            'DESCRIPTION': atc.get('level4_description'),
            'PARENT_CODE': atc.get('level3')
        })
    
    # Level 5
    code = atc.get('level5')
    if code and code not in seen_codes:
        seen_codes.add(code)
        atc_records.append({
            'ATC_CODE': code,
            'LEVEL': 5,
            'DESCRIPTION': atc.get('who_name'),
            'PARENT_CODE': atc.get('level4')
        })

df_atc_class = pd.DataFrame(atc_records)

print(f'ATC hierarchy master: {len(df_atc_class):,} records')
print(f'\nRecords by level:')
print(df_atc_class['LEVEL'].value_counts().sort_index())

df_atc_class.head(20)

In [ ]:
# Save local backup
df_atc_class.to_parquet(results_dir / 'atc_classification.parquet', index=False)
print(f'Saved: {results_dir / "atc_classification.parquet"}')

## 3. Snowflake Connection

In [ ]:
def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('~/.ssh/snowflake_rsa_key.pem')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der

def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user="KOREEDA",
            account="DUETMBM-LL33279",
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

conn = connect_to_snowflake()

## 4. Create ATC_COMPOUND_MAP Table

In [ ]:
cursor = conn.cursor()

# Drop table
table_name = "ATC_COMPOUND_MAP"
drop_query = f"DROP TABLE IF EXISTS BIOINFORMATICS.LINCS.{table_name}"
cursor.execute(drop_query)
print(f'Dropped (if exists): {table_name}')

# Create table
create_query = f"""
CREATE TABLE BIOINFORMATICS.LINCS.{table_name} (
    CHEMBL_ID VARCHAR(20),
    INCHI_KEY VARCHAR(30),
    CANONICAL_SMILES VARCHAR(4000),
    ATC_CODE VARCHAR(10),
    ATC_L1 VARCHAR(1),
    ATC_L2 VARCHAR(3),
    ATC_L3 VARCHAR(4),
    ATC_L4 VARCHAR(5),
    PREF_NAME VARCHAR(500),
    MAX_PHASE INT
)
"""
cursor.execute(create_query)
print(f'Created: {table_name}')

In [ ]:
# Data upload (using write_pandas)
from snowflake.connector.pandas_tools import write_pandas

print(f'Uploading {len(df_atc_map):,} records to {table_name}...')

# Convert NaN to None and MAX_PHASE to int type
df_upload = df_atc_map.copy()
df_upload['MAX_PHASE'] = pd.to_numeric(df_upload['MAX_PHASE'], errors='coerce').fillna(0).astype(int)

# Upload with write_pandas
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df_upload,
    table_name=table_name,
    database='BIOINFORMATICS',
    schema='LINCS',
    quote_identifiers=False
)

print(f'Upload complete: {nrows:,} rows in {nchunks} chunks')

## 5. Create ATC_CLASSIFICATION Table

In [ ]:
# Drop table
table_name = "ATC_CLASSIFICATION"
drop_query = f"DROP TABLE IF EXISTS BIOINFORMATICS.LINCS.{table_name}"
cursor.execute(drop_query)
print(f'Dropped (if exists): {table_name}')

# Create table
create_query = f"""
CREATE TABLE BIOINFORMATICS.LINCS.{table_name} (
    ATC_CODE VARCHAR(10),
    LEVEL INT,
    DESCRIPTION VARCHAR(500),
    PARENT_CODE VARCHAR(10)
)
"""
cursor.execute(create_query)
print(f'Created: {table_name}')

In [ ]:
# Data upload (using write_pandas)
print(f'Uploading {len(df_atc_class):,} records to {table_name}...')

df_upload = df_atc_class.copy()

# Upload with write_pandas
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df_upload,
    table_name=table_name,
    database='BIOINFORMATICS',
    schema='LINCS',
    quote_identifiers=False
)

print(f'Upload complete: {nrows:,} rows in {nchunks} chunks')

## 6. LINCS Join Test

In [ ]:
# Check coverage by joining with LINCS InChIKeys
test_query = """
SELECT 
    COUNT(DISTINCT l."inchi_key") as lincs_total,
    COUNT(DISTINCT CASE WHEN a.INCHI_KEY IS NOT NULL THEN l."inchi_key" END) as atc_matched
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE l
LEFT JOIN BIOINFORMATICS.LINCS.ATC_COMPOUND_MAP a
  ON l."inchi_key" = a.INCHI_KEY
WHERE l."inchi_key" IS NOT NULL
  AND l."inchi_key" != ''
"""

df_test = pd.read_sql(test_query, conn)
print('='*60)
print('LINCS x ATC_COMPOUND_MAP Join Results')
print('='*60)

lincs_total = df_test['LINCS_TOTAL'].iloc[0]
atc_matched = df_test['ATC_MATCHED'].iloc[0]
coverage = atc_matched / lincs_total * 100 if lincs_total > 0 else 0

print(f'LINCS unique InChIKeys: {lincs_total:,}')
print(f'ATC matches: {atc_matched:,}')
print(f'Coverage: {coverage:.1f}%')

In [ ]:
# Check ATC class (L2) distribution
dist_query = """
SELECT 
    a.ATC_L2,
    c.DESCRIPTION,
    COUNT(DISTINCT l."inchi_key") as n_compounds
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE l
INNER JOIN BIOINFORMATICS.LINCS.ATC_COMPOUND_MAP a
  ON l."inchi_key" = a.INCHI_KEY
LEFT JOIN BIOINFORMATICS.LINCS.ATC_CLASSIFICATION c
  ON a.ATC_L2 = c.ATC_CODE AND c.LEVEL = 2
WHERE l."inchi_key" IS NOT NULL
  AND l."inchi_key" != ''
GROUP BY a.ATC_L2, c.DESCRIPTION
ORDER BY n_compounds DESC
"""

df_dist = pd.read_sql(dist_query, conn)
print('\nATC class (L2) distribution (LINCS compounds):')
print(df_dist.head(20).to_string(index=False))

In [ ]:
# Close connection
conn.close()
print('\nSnowflake connection closed')
print('Complete!')

## Summary

### Created Tables
1. `BIOINFORMATICS.LINCS.ATC_COMPOUND_MAP` - Mapping between compounds and ATC codes
2. `BIOINFORMATICS.LINCS.ATC_CLASSIFICATION` - ATC code hierarchy and description master

### INCHI_KEY_14 Column (for partial matching)

InChIKey consists of 27 characters divided into 3 parts:
- **First 14 characters**: Represents molecular skeleton (structure) (e.g., `QFJCIRLUMZQUOT`)
- **Next 10 characters**: Stereochemistry information (e.g., `XWBLMERNSA`)
- **Last 1 character**: Protonation state (e.g., `N`)

Since different stereoisomer registrations of the same compound may have different InChIKeys,
partial matching using the **first 14 characters (INCHI_KEY_14)** is recommended.

#### Matching Improvement (Liver Cancer Cell Lines)
| Matching Method | Matches | Coverage |
|-----------------|---------|----------|
| Exact match | 762 | 16.2% |
| Partial match (first 14 chars) | 944 | 20.1% |

**+182 compounds (+23.9%)** improvement

### Usage

#### Recommended: Partial Match (First 14 Characters)
```sql
SELECT g.*, a.ATC_CODE, a.ATC_L2, c.DESCRIPTION
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE g
LEFT JOIN BIOINFORMATICS.LINCS.ATC_COMPOUND_MAP a
  ON SUBSTRING(g."inchi_key", 1, 14) = a.INCHI_KEY_14
LEFT JOIN BIOINFORMATICS.LINCS.ATC_CLASSIFICATION c
  ON a.ATC_L2 = c.ATC_CODE AND c.LEVEL = 2
```

#### Traditional: Exact Match
```sql
SELECT g.*, a.ATC_CODE, a.ATC_L2, c.DESCRIPTION
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE g
LEFT JOIN BIOINFORMATICS.LINCS.ATC_COMPOUND_MAP a
  ON g."inchi_key" = a.INCHI_KEY
LEFT JOIN BIOINFORMATICS.LINCS.ATC_CLASSIFICATION c
  ON a.ATC_L2 = c.ATC_CODE AND c.LEVEL = 2
```